# 🫁 CheXpert Scientific Protocol: Kaggle Dual-GPU (2x Tesla T4) Training

This notebook is optimized for Kaggle's **Dual Tesla T4 GPUs (2x 16GB independent VRAM)** to train deep convolutional architectures (`convnext_small` or `densenet121`) under **Protocol v0.1**.

### ⚡ Dual-GPU & Protocol Configurations
- **PyTorch DataParallel**: Distributes batches across both independent 16GB T4 GPUs.
- **Fail-Closed Verification**: Automatic integrity checks on splits, dataset, and checkpoints.
- **Patient-Level Segregation**: 80% train, 10% val, 10% calib strictly partitioned at the patient level with zero leakage.
- **Independent Threshold Calibration**: F1 optimization solely on the dedicated calibration split.


In [ ]:
# ==============================================================================
# ⚙️ EXPERIMENT & HARDWARE CONFIGURATION
# ==============================================================================
ARCH = "convnext_small"
SEED = 42
RUN_MODE = "smoke"
REPO_REF = "main"
DATA_ROOT = None
BATCH_SIZE = 32
RESUME_CHECKPOINT = None
RESUME_CHECKSUMS_JSON = None
RESUME_BUNDLE = None
EXPECTED_RESUME_BUNDLE_SHA256 = None  # externally recorded producer digest
EXPECTED_SOURCE_CSV_SHA256 = None
SMOKE_PHASE = "fresh"  # "fresh" or "resume"

def require(condition, message, exc_type=RuntimeError):
    if not condition:
        raise exc_type(message)

# Enforce single-model, single-seed execution
require(ARCH in ["convnext_small", "densenet121"], f"Invalid ARCH: {ARCH}", ValueError)
require(SEED in [42, 43, 44, 45, 46], f"Invalid SEED: {SEED}", ValueError)
require(RUN_MODE in ["smoke", "full"], f"Invalid RUN_MODE: {RUN_MODE}", ValueError)
require(SMOKE_PHASE in ["fresh", "resume"], f"Invalid SMOKE_PHASE: {SMOKE_PHASE}", ValueError)
require(BATCH_SIZE == 32, f"Default batch size must be 32, got {BATCH_SIZE}", ValueError)

# Strict REPO_REF policy:
# Only "smoke" mode permits tracking "main".
# In "full" scientific mode, REPO_REF MUST be a pinned commit SHA or release tag.
if RUN_MODE == "full":
    if REPO_REF in ["main", "master"] or not REPO_REF:
        raise ValueError(
            f"PROTOCOL REPRODUCIBILITY VIOLATION: In RUN_MODE='full', REPO_REF must be a pinned commit SHA "
            f"or release tag (got '{REPO_REF}'). Tracking 'main' or 'master' is strictly prohibited in official protocol runs."
        )

WORK_DIR = "/kaggle/working/chex"
print(f"[CONFIG] Arch: {ARCH} | Seed: {SEED} | Mode: {RUN_MODE} | Smoke Phase: {SMOKE_PHASE} | Batch: {BATCH_SIZE} | Repo Ref: {REPO_REF}")


In [ ]:
# ==============================================================================
# 🚀 CELL 1: Dual-GPU & System Diagnostic
# ==============================================================================
import os
import sys
import torch
import torchvision

print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs Detected    : {gpu_count}")

for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  -> GPU {i}: {name} ({vram:.2f} GB independent VRAM)")

if gpu_count >= 2:
    print("🔥 [OPTIMIZED] Dual-GPU DataParallel mode ready across independent 16GB VRAM devices.")
elif gpu_count == 1:
    print("⚡ [OPTIMIZED] Single GPU mode active.")
else:
    if RUN_MODE == "full":
        raise RuntimeError("HARDWARE ERROR: Full protocol training requires GPU accelerator! Please enable GPU in Kaggle Settings.")
    print("⚠️ [WARNING] No GPU detected! Running smoke check on CPU.")


In [ ]:
# ==============================================================================
# 📥 CELL 2: Repository Setup & Code Synchronization
# ==============================================================================
import os
import subprocess
from pathlib import Path

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

if not (Path(WORK_DIR) / ".git").exists():
    print(f"Cloning repository from GitHub into {WORK_DIR}...")
    subprocess.check_call(["git", "clone", "https://github.com/qdat2644/chex.git", "."])
else:
    print(f"Pulling latest code updates in {WORK_DIR}...")
    subprocess.check_call(["git", "fetch", "origin"])

subprocess.check_call(["git", "checkout", REPO_REF])
if REPO_REF == "main":
    subprocess.check_call(["git", "pull", "origin", "main"])
else:
    subprocess.check_call(["git", "status"])

commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"[OK] Repository active at Git commit: {commit_sha}")


In [ ]:
# ==============================================================================
# 📦 CELL 3: Dependencies Installation & Environment Compilation
# ==============================================================================
import subprocess
import sys

print("Installing requirements from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"])

# Verify core project modules
import torch, torchvision, pydicom, sklearn, scipy, yaml, pandas, numpy, PIL, fastapi, uvicorn
print(f"[OK] Core libraries verified: PyTorch {torch.__version__}, SciPy {scipy.__version__}, PyDICOM {pydicom.__version__}")

subprocess.check_call([sys.executable, "-m", "compileall", "-q", "app", "scripts", "tests"])
print("[OK] Codebase verified and compiled.")


In [ ]:
# ==============================================================================
# 🔍 CELL 4: Dataset Discovery & Image Integrity Verification
# ==============================================================================
from pathlib import Path
import pandas as pd

input_root = Path('/kaggle/input')
print(f'Searching for CheXpert train.csv in {input_root}...')

from scripts.kaggle_preflight import (
    resolve_unique_data_root, select_train_csv, validate_development_source,
    validate_git_integrity, verify_expected_source_csv,
)

train_csv_path = select_train_csv(input_root, Path(DATA_ROOT) if DATA_ROOT else None)
data_root_dir = train_csv_path.parent
candidate_roots = ([Path(DATA_ROOT), Path(DATA_ROOT) / 'CheXpert-v1.0-small'] if DATA_ROOT else []) + [
    data_root_dir.parent, data_root_dir,
    data_root_dir.parent / 'CheXpert-v1.0-small',
    data_root_dir / 'CheXpert-v1.0-small',
]
resolved_data_root = resolve_unique_data_root(train_csv_path, candidate_roots, sample_size=50)
source_df = pd.read_csv(train_csv_path)
require('Path' in source_df.columns, f'Missing Path column in {train_csv_path}', ValueError)
validate_development_source(resolved_data_root, train_csv_path, source_df['Path'])
source_csv_sha256 = verify_expected_source_csv(train_csv_path, RUN_MODE, EXPECTED_SOURCE_CSV_SHA256)
git_integrity = validate_git_integrity(Path(WORK_DIR), REPO_REF, RUN_MODE)
print(f'[FOUND] Training CSV : {train_csv_path}')
print(f'[FOUND] Data Root    : {resolved_data_root}')
print(f'[PASSED] Verified 50 image paths, source SHA-256, source role, and Git integrity.')


In [ ]:
# ==============================================================================
# 🛡️ CELL 5: Patient-Level Partitioning (Zero Leakage Split)
# ==============================================================================
import subprocess
import sys
from pathlib import Path

manifest_dir = Path(WORK_DIR) / "outputs" / "splits" / "protocol_v0_1"
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_dir / "manifest.json"

if not manifest_path.is_file():
    print("Generating patient-level split (80% train, 10% val, 10% calib) from training cohort...")
    split_cmd = [
        sys.executable, "scripts/make_splits.py",
        "--data-root", str(resolved_data_root),
        "--train-csv", str(train_csv_path),
        "--output-dir", str(manifest_dir),
        "--seed", "42",
        "--protocol-version", "0.1",
    ]
    subprocess.check_call(split_cmd)
else:
    print(f"Using existing manifest: {manifest_path}")


In [ ]:
# ==============================================================================
# 🔒 CELL 6: Split Anti-Leakage & Prevalence Verification
# ==============================================================================
import hashlib
import json
import pandas as pd
from pathlib import Path

manifest_data = json.loads(manifest_path.read_text(encoding='utf-8'))
splits_info = manifest_data.get('splits', {})
train_csv_file = manifest_dir / splits_info['train']['csv']
cal_csv_file = manifest_dir / splits_info['calibration']['csv']
val_csv_file = manifest_dir / splits_info['internal_validation']['csv']

train_df = pd.read_csv(train_csv_file)
cal_df = pd.read_csv(cal_csv_file)
val_df = pd.read_csv(val_csv_file)

train_pids = set(train_df['patient_id'])
val_pids = set(val_df['patient_id'])
cal_pids = set(cal_df['patient_id'])

# 1. Zero patient overlap verification
require(len(train_pids & val_pids) == 0, 'CRITICAL: Train and Validation patient overlap!')
require(len(train_pids & cal_pids) == 0, 'CRITICAL: Train and Calibration patient overlap!')
require(len(val_pids & cal_pids) == 0, 'CRITICAL: Validation and Calibration patient overlap!')

# 2. Duplicate image hash check across splits
from scripts.kaggle_preflight import validate_split_image_hashes
total_duplicate_hashes = validate_split_image_hashes({
    'train': train_df, 'calibration': cal_df, 'internal_validation': val_df,
})

# Duplicate image/study path check across splits
path_col = 'Path' if 'Path' in train_df.columns else 'image_path'
train_paths = set(train_df[path_col])
val_paths = set(val_df[path_col])
cal_paths = set(cal_df[path_col])
require(len(train_paths & val_paths) == 0, 'CRITICAL: Duplicate image path between Train and Validation!')
require(len(train_paths & cal_paths) == 0, 'CRITICAL: Duplicate image path between Train and Calibration!')
require(len(val_paths & cal_paths) == 0, 'CRITICAL: Duplicate image path between Validation and Calibration!')

print(f'[PASSED] Zero patient overlap and zero duplicate hashes across splits.')
print(f'  - Patient overlap       : 0')
print(f'  - Duplicate hashes      : {total_duplicate_hashes}')
print(f'  - Train split           : {len(train_pids)} patients, {len(train_df)} images')
print(f'  - Internal Val split    : {len(val_pids)} patients, {len(val_df)} images')
print(f'  - Calibration split     : {len(cal_pids)} patients, {len(cal_df)} images')

# 3. Print Label Prevalences
labels = manifest_data.get('labels', ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion'])
print('\n--- Label Prevalence Across Splits ---')
for lbl in labels:
    p_tr = (train_df[lbl] == 1.0).mean() * 100
    p_vl = (val_df[lbl] == 1.0).mean() * 100
    p_cl = (cal_df[lbl] == 1.0).mean() * 100
    print(f'  {lbl:18s} -> Train: {p_tr:5.1f}% | Val: {p_vl:5.1f}% | Calib: {p_cl:5.1f}%')

manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
print(f'\n[PASSED] Manifest SHA-256: {manifest_hash}')


In [ ]:
# ==============================================================================
# 🔥 CELL 7: Protocol Training Execution
# ==============================================================================
import subprocess
import sys
import torch
import yaml
from pathlib import Path

out_run_dir = Path(WORK_DIR) / "outputs" / "runs" / ARCH / f"seed_{SEED}"
config_file = Path(WORK_DIR) / "configs" / "protocol_v0_1.yaml"

# Load protocol config
cfg = yaml.safe_load(config_file.read_text(encoding="utf-8"))
t_cfg = cfg.get("training", {})

workers = "4" if torch.cuda.is_available() else "0"
train_cmd = [
    sys.executable, "scripts/train.py",
    "--manifest", str(manifest_path),
    "--config", str(config_file),
    "--data-root", str(resolved_data_root),
    "--arch", ARCH,
    "--seed", str(SEED),
    "--output-dir", str(out_run_dir),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", workers,
]

if RUN_MODE == "smoke":
    if SMOKE_PHASE == "fresh":
        print("Running FRESH SMOKE TEST: Planned 2 epochs, stopping after epoch 1 (128 samples)...")
        train_cmd.extend(["--epochs", "2", "--stop-after-epoch", "1", "--limit", "128", "--run-mode", "smoke"])
    else:
        print("Running RESUME SMOKE TEST: Planned 2 epochs, resuming from checkpoint to complete epoch 2...")
        require(RESUME_BUNDLE is not None, 'RESUME_BUNDLE is mandatory for resume smoke')
        require(EXPECTED_RESUME_BUNDLE_SHA256, 'External expected bundle SHA-256 is mandatory')
        verified_resume_dir = Path(WORK_DIR) / 'verified_resume_bundle'
        subprocess.check_call([sys.executable, 'scripts/verify_resume_bundle.py', '--bundle', str(RESUME_BUNDLE), '--expected-sha256', EXPECTED_RESUME_BUNDLE_SHA256, '--extract-dir', str(verified_resume_dir)])
        resume_ledger = json.loads((verified_resume_dir / 'checksums.json').read_text(encoding='utf-8'))
        resume_target = verified_resume_dir / 'last.pt'
        expected_resume_sha256 = resume_ledger['files']['last.pt']
        train_cmd.extend(["--epochs", "2", "--resume", str(resume_target), "--expected-resume-sha256", expected_resume_sha256, "--limit", "128", "--run-mode", "smoke"])
else:
    print(f"Running FULL PROTOCOL TRAINING (20 epochs, ASL loss, AdamW, Cosine, Arch={ARCH}, Seed={SEED}, Batch={BATCH_SIZE})...")
    train_cmd.extend(["--run-mode", "full"])
    if RESUME_CHECKPOINT:
        require(RESUME_BUNDLE and EXPECTED_RESUME_BUNDLE_SHA256, 'Full resume requires bundle and external expected SHA-256')
        from scripts.verify_resume_bundle import verify_resume_bundle
        full_resume_ledger = verify_resume_bundle(Path(RESUME_BUNDLE), expected_sha256=EXPECTED_RESUME_BUNDLE_SHA256)
        expected_resume_sha256 = full_resume_ledger['files']['last.pt']
        train_cmd.extend(["--resume", str(RESUME_CHECKPOINT), "--expected-resume-sha256", expected_resume_sha256])

subprocess.check_call(train_cmd)

# Verify outputs
if RUN_MODE == "smoke" and SMOKE_PHASE == "fresh":
    require((out_run_dir / "last.pt").is_file(), "Fresh smoke must create last.pt!", FileNotFoundError)
    require((out_run_dir / "best.pt").is_file(), "Fresh smoke must create best.pt from epoch 1!", FileNotFoundError)
    resume_bundle_path = Path('/kaggle/working/kaggle_artifacts') / f'{ARCH}_seed{SEED}_resume.zip'
    subprocess.check_call([sys.executable, 'scripts/package_resume_bundle.py', '--last-checkpoint', str(out_run_dir / 'last.pt'), '--resolved-config', str(out_run_dir / 'resolved_config.json'), '--manifest', str(manifest_path), '--protocol-config', str(config_file), '--output', str(resume_bundle_path)])
    print(f"[SUCCESS] Fresh smoke resume bundle created and verified -> {resume_bundle_path}")
else:
    required_run_outputs = [
        out_run_dir / "best.pt",
        out_run_dir / "last.pt",
        out_run_dir / "training_history.json",
        out_run_dir / "run_manifest.json",
        out_run_dir / "resolved_config.json",
        out_run_dir / "internal_validation_predictions.csv",
    ]
    for req_file in required_run_outputs:
        if not req_file.is_file():
            raise FileNotFoundError(f"MANDATORY OUTPUT MISSING: {req_file}")
    print(f"\n[SUCCESS] Training completed and all 6 artifacts verified -> {out_run_dir}")


In [ ]:
# ==============================================================================
# 🎯 CELL 8: Finding-Specific Threshold Calibration
# ==============================================================================
import subprocess
import sys
from pathlib import Path

if RUN_MODE == "smoke" and SMOKE_PHASE == "fresh":
    print("Skipping calibration in fresh smoke phase (switch to SMOKE_PHASE='resume' to test calibration & packaging).")
else:
    calib_dir = Path(WORK_DIR) / "outputs" / "calibration"
    calib_dir.mkdir(parents=True, exist_ok=True)
    calib_out = calib_dir / f"{ARCH}_seed{SEED}.json"
    best_ckpt = out_run_dir / "best.pt"

    print(f"Optimizing F1 thresholds on Calibration Split for {ARCH} seed {SEED}...")
    calib_cmd = [
        sys.executable, "scripts/calibrate.py",
        "--checkpoint", str(best_ckpt),
        "--split-manifest", str(manifest_path),
        "--config", str(config_file),
        "--data-root", str(resolved_data_root),
        "--output", str(calib_out),
        "--arch", ARCH,
        "--seed", str(SEED),
        "--run-mode", RUN_MODE,
    ]
    if RUN_MODE == "smoke":
        calib_cmd.extend(["--limit", "128"])

    subprocess.check_call(calib_cmd)
    if not calib_out.is_file():
        raise FileNotFoundError(f"Missing calibration artifact at {calib_out}")

    print(f"[SUCCESS] Calibration artifact produced and verified -> {calib_out}")


In [ ]:
# ==============================================================================
# 📦 CELL 9: Artifact Packaging & Integrity Checksum Ledger
# ==============================================================================
import subprocess
import sys
from pathlib import Path

if RUN_MODE == "smoke" and SMOKE_PHASE == "fresh":
    print("Skipping packaging in fresh smoke phase (switch to SMOKE_PHASE='resume' to test packaging).")
else:
    pkg_dir = Path("/kaggle/working/kaggle_artifacts")
    pkg_dir.mkdir(parents=True, exist_ok=True)
    zip_path = pkg_dir / f"{ARCH}_seed{SEED}.zip"

    print(f"Packaging and verifying artifacts into {zip_path}...")
    package_cmd = [
        sys.executable, "scripts/package_kaggle_artifact.py",
        "--run-dir", str(out_run_dir),
        "--calibration", str(calib_out),
        "--manifest", str(manifest_path),
        "--config", str(config_file),
        "--output", str(zip_path),
        "--expected-arch", ARCH,
        "--expected-seed", str(SEED),
        "--expected-run-mode", RUN_MODE,
    ]
    subprocess.check_call(package_cmd)
    print(f"[SUCCESS] Packaging and post-verification complete: {zip_path}")
